# WS6 — Learned Representations: A Capacity-Matched GRU Ablation

**Workstream 6 of the pitch-sequencing rigor ladder — the deep-learning rung and the study's
compute peak on the predictive side.** This notebook fits and reads a small **gated recurrent
network** (GRU) across **all five** nested state views, for both a next-pitch **selection** model
and a current-action-conditioned **outcome** model. It is the rung that stops hand-crafting history
features and lets the net read the plate appearance like a sentence.

## The one question WS6 asks

WS1 was a lookup table; WS2 wrote the ordered rule down in an **explicit grammar**; WS3 fed a strong
tree model **engineered** history features (positional slots, velocity/location differences, D11).
WS6 stops engineering:

> When we stop hand-crafting features and let a small neural net read the at-bat like a sentence —
> one word per prior pitch — does it find ordered signal the **engineered** features (WS3) and the
> **explicit** grammar (WS2) missed?

## The design principle, up front: vary INFORMATION, not CAPACITY (decision D45)

A naive "deep beats tabular" comparison confounds the learned representation with the extra
parameters. WS6 removes that confound by realising the five views as **capacity-matched
architectures**:

- **C** — a static context MLP (no sequence branch);
- **U** — mean-pooled token embeddings, **order-invariant by construction**;
- **L1** — a GRU truncated to the **last** history token;
- **O** — the **same** GRU over the **full ordered** sequence;
- **OM** — O + a matchup-memory branch.

The clean pair is **L1 vs O**: the *identical* network, differing only in how much of the sequence
is fed in. Their parameter counts are equal by construction (the committed run: `31,112` each), so
any O-over-L1 gain is *ordered information*, not model size. A bigger O would confound information
with capacity — the exact failure D45 exists to prevent.

## The headline comparison (SPEC §6, verbatim)

> Every suitable model is trained on the **same five state variants**. The differences between them
> *are* the sequencing evidence.
>
> $$\Delta_{\text{order}} = \text{Loss}(\min[U, L1]) - \text{Loss}(O), \qquad
>   \Delta_{\text{matchup}} = \text{Loss}(O) - \text{Loss}(OM)$$
>
> Interpretation: O beating C but **not** U/L1 ⇒ history matters but *order* barely does. O beating
> both U and L1 ⇒ genuine ordered dependence.

WS6 answers this on **both** targets — next-pitch **selection** and current-action-conditioned
**outcome** — with pitcher-game clustered CIs on validation (2024) and the locked test (2025), plus
the capacity-matched twin gate $\Delta_{\text{twin}} = \text{Loss}(L1) - \text{Loss}(O)$ (equal
parameters, so a pure order comparison) read on the subset the effect acts on (§6).

## The DATA_MODE toggle

This notebook is a **scaffold**. Phase 2 runs it on the real Statcast decision table; here a single
toggle, `DATA_MODE`, selects the world:

- `'synth_null'` — the oracle's **null world** (an ordered *selection* habit — a no-three-in-a-row
  tendency — but no ordered *outcome* effect). **Default**, because WS6's headline exhibits — grammar
  detection, the targeting lesson, and the opacity probe — all live here.
- `'synth_positive'` — the oracle's **positive world** (a planted velocity-transition whiff boost),
  the outcome-mechanism recovery control.
- `'real'` — the real decision table built by `python -m pitchseq.build_table` (Phase 2).

## The three findings this notebook keeps separate (SPEC §0, verbatim)

> 1. **Selection structure** — prior pitches help predict *what is thrown next*.
> 2. **Predictive sequencing value** — prior pitches help predict the *outcome* of the current pitch,
>    after conditioning on the current pitch and game state.
> 3. **Prescriptive/causal value** — *changing* the sequence would improve outcomes.

WS6 lives in **findings #1 and #2**: a learned predictor is *predictive*, not causal. Turning that
into finding #3 is the OPE/RL workstreams' job — the firewall we restate throughout.

## How to read this notebook

Every code step is bracketed by plain-worded markdown: **before** each cell we say what will happen
and why; **after** each cell we say how to read what came out. Numbers that depend on the real data
are `{PLACEHOLDER}` in the companion `PAPER.md`; here they simply appear when you run the cell (the
cross-workstream WS2/WS3 losses are a *read-off* — pasted from their reports). The **Results** section
(§9) is *branched* on three axes — representation (R+/R=/R−) × order (H1/H2/H3) × matchup (M+/M0/M−):
a code cell inspects the computed numbers and prints which branch applies, and the markdown that
follows holds the pre-written interpretation for every branch. The exact formulas live in `THEORY.md`;
the exact code in `workstreams/ws6_deep_seq/model.py` and `run_ws6.py`; the plain-English tour in
`SEAN-README.md`.

> **Compute note (SPEC §7 Pareto discipline).** WS6 is the study's compute peak on the predictive
> side and its **only GPU-relevant** workstream. PyTorch lives in the `[deep]` extra
> (`pip install -e ".[deep]"`); the code is device-agnostic (`--device auto` → CUDA if present, else
> CPU). The recurrent O/OM fits dominate, so every fit logs params / wall-clock / peak RAM for the
> performance-vs-compute plot, and **L1 and O print identical parameter counts** — the capacity
> match, so the O premium is pure wall-clock. The in-notebook budgets (`NB_N_GAMES`, `NB_EPOCHS`, the
> falsification budgets) are collected in the setup cell for a quick scaffold pass — the committed
> validation used `n_games=200`, 6 epochs (~28 min); Phase-2 real runs go through the CLI (RUNBOOK
> WS6.1 local CPU / WS6.2 free Colab T4), not this notebook.

## 1. Setup

In [ ]:
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- locate the repository root (works from repo root or from notebooks/) ---
REPO_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "pyproject.toml").exists() and (_p / "workstreams").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- shared foundation (WS0) ---
from pitchseq.config import load_config
from pitchseq.families import FAMILIES
from pitchseq.outcomes import OUTCOME1
from pitchseq.sequences import build_sequences, SEQUENCE_FLOAT_CHANNELS
from pitchseq.decision_table import build_decision_table
from pitchseq.synth import make_null_world, make_positive_world
from pitchseq.eval.metrics import log_loss_per_row

# --- WS6 (this workstream) -- torch is the '.[deep]' extra: pip install -e ".[deep]" ---
import torch
from workstreams.ws6_deep_seq.model import (
    STATE_VIEWS, TARGETS, FeatureEncoders, SeqModel, _SeqNet,
    seed_everything, resolve_device, make_ws6_model_factory, motif_repeat_probe,
    temporal_split_masks, ws6_order_ablation, _repeat_context_mask,
    _l1_twin_delta_ci, _delta_order_ci, _target_labels, _target_values,
    _FAM_INDEX, _O1_INDEX,
)
from workstreams.ws6_deep_seq.run_ws6 import run_ws6, _format_headline, _DEFAULT_HP

CONFIG = load_config()
SEED = int(CONFIG.get("seeds", {}).get("global", 20260713))

# The world this run analyses: 'synth_null' | 'synth_positive' | 'real'.
# Default 'synth_null': it plants an ordered SELECTION habit (no-three-in-a-row) but no ordered
# OUTCOME effect, so WS6's headline exhibits -- grammar detection, the targeting lesson, and the
# opacity probe -- all fire here (the positive world is the outcome-mechanism control).
DATA_MODE = "synth_null"
_SYNTH = {"synth_null": "null", "synth_positive": "positive", "real": "off"}

# WS6 runs ALL FIVE views (D45) as capacity-matched nets, on both targets.
VIEWS = ["C", "U", "L1", "O", "OM"]
TARGETS_ = ["selection", "outcome1"]

# The compact D46 architecture (identical across views -- only the fed information differs).
EMBED, HIDDEN, MAX_LEN, BATCH = 16, 64, 15, 512

# --- in-notebook budgets (a quick scaffold pass; the committed run used n_games=200, epochs=6) ---
NB_N_GAMES = 120     # synthetic-world size (ignored when DATA_MODE == 'real')
NB_EPOCHS = 4        # max training epochs (early-stopped on val); committed validation used 6
NB_N_PERM = 6        # token-order permutation refits (synthetic-only D47 gate)
NB_N_BOOT = 120      # cluster-bootstrap replicates for the ablation CIs (width only)
NB_CI_BOOT = 40      # falsification CI replicates
WORLD_SEED = 7       # seed for the synthetic world + the fits/bootstrap in this notebook

# Phase-2 real input (Step 1 builds the decision table; RUNBOOK WS6.1/WS6.2).
REAL_TABLE_PATH = REPO_ROOT / "data" / "processed" / "decision_table.parquet"

# The committed-validation Pareto rows (null world, 6 epochs, ~1.0 GB peak) -- the honest compute
# table (D13/SPEC §7). Live values reappear from the pipeline report in section 5.
COMMITTED_PARETO = {  # view: (n_params, seconds)
    "C": (6856, 62.6), "U": (13512, 74.8), "L1": (31112, 81.1),
    "O": (31112, 145.9), "OM": (36616, 110.1),
}

print(f"repo root : {REPO_ROOT}")
print(f"DATA_MODE : {DATA_MODE}   ->  run_ws6 synth='{_SYNTH[DATA_MODE]}'")
print(f"views     : {VIEWS}   targets: {TARGETS_}   (all five, both targets; D45/D22)")
print(f"arch      : embed={EMBED} hidden={HIDDEN} max_len={MAX_LEN}  (compact, D46; L1 and O share it)")
print(f"budgets   : n_games={NB_N_GAMES} epochs={NB_EPOCHS} n_perm={NB_N_PERM}")
print(f"seed      : {SEED}   world_seed: {WORLD_SEED}")
print(f"torch     : {torch.__version__}   device(auto): {resolve_device('auto')}   cuda: {torch.cuda.is_available()}")

### Plotting style (fixed, colorblind-safe view colours)

In [ ]:
# Okabe-Ito qualitative palette (colorblind-safe) -- identical to WS1/WS2/WS3/WS4/WS5.
OKABE_ITO = {
    "orange":         "#E69F00",
    "sky_blue":       "#56B4E9",
    "bluish_green":   "#009E73",
    "yellow":         "#F0E442",
    "blue":           "#0072B2",
    "vermillion":     "#D55E00",
    "reddish_purple": "#CC79A7",
    "black":          "#000000",
}
# Fixed view -> colour, IDENTICAL to WS3: C context base, O the fully ordered headline, OM + matchup.
VIEW_COLORS = {
    "C":  OKABE_ITO["blue"],
    "U":  OKABE_ITO["orange"],
    "L1": OKABE_ITO["bluish_green"],
    "O":  OKABE_ITO["vermillion"],
    "OM": OKABE_ITO["reddish_purple"],
}
REF_COLOR = OKABE_ITO["black"]  # neutral: count-based references, the capacity-match reference line

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "figure.autolayout": True,
})


def style_axes(ax):
    """Left+bottom spines only; no top/right. Returns the axis for chaining."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax


def new_fig(figsize=(7.2, 4.2)):
    """One figure, one axis, pre-styled."""
    fig, ax = plt.subplots(figsize=figsize)
    style_axes(ax)
    return fig, ax

## 2. Data → tensors — what the GRU sees that the tabular `O` sees only through slots

WS6 consumes the same five nested views (SPEC §6) but through their **raw sequence form**. For every
decision row, `build_sequences` (SPEC §3.3) returns the ordered list of *prior* pitches this PA,
oldest-first, left-aligned and padded to `max_len = 15`, each position carrying a **family** token, an
**outcome** token, and seven **float channels**
(`plate_x_br, plate_z_norm, release_speed, pfx_x, pfx_z, dvelo, dloc`) — with `dvelo`/`dloc` the
release-speed and location *changes* from the previous history pitch. Only prior pitches' realized
(`exec_`) values enter, so it is leakage-safe (SPEC §0). A boolean `mask`, per-row `lengths`, and a
`row_id` array keep the tensors aligned to the decision table. This is exactly the material a tree
view reads only through hand-designed positional slots (D11): the GRU sees the raw ordered
`(family, outcome, physics, transitions)` per prior pitch and learns its own summary.

In [ ]:
def load_world(mode):
    """Build (synthetic) or load (real) the decision table + its ground-truth dict for DATA_MODE."""
    if mode == "real":
        return pd.read_parquet(REAL_TABLE_PATH), {}
    if mode == "synth_positive":
        raw, truth = make_positive_world(n_games=NB_N_GAMES, seed=WORLD_SEED,
                                         effect_size=0.30, velo_gap_threshold=5.0)
    else:
        raw, truth = make_null_world(n_games=NB_N_GAMES, seed=WORLD_SEED)
    return build_decision_table(raw), truth


table, TRUTH = load_world(DATA_MODE)
masks = temporal_split_masks(table, CONFIG)
n_tr, n_va, n_te = int(masks["train"].sum()), int(masks["val"].sum()), int(masks["test"].sum())

seq = build_sequences(table, max_len=MAX_LEN)
tok_dim = 2 * EMBED + len(SEQUENCE_FLOAT_CHANNELS)   # family + outcome embeddings + float channels
print(f"world               : {DATA_MODE}   rows: train={n_tr:,} val={n_va:,} test={n_te:,}")
print(f"sequence tensors    : family_idx {seq['family_idx'].shape}  outcome1_idx {seq['outcome1_idx'].shape}")
print(f"float channels ({len(SEQUENCE_FLOAT_CHANNELS)}) : {list(SEQUENCE_FLOAT_CHANNELS)}")
print(f"pad indices         : family_pad={seq['family_pad_idx']}  outcome1_pad={seq['outcome1_pad_idx']}")
print(f"token width d_tok   : 2*{EMBED} (fam+out embeddings) + {len(SEQUENCE_FLOAT_CHANNELS)} channels = {tok_dim}")
print(f"row_id aligned      : {seq['row_id'].shape[0]:,} rows == len(table) {len(table):,}  ->  {seq['row_id'].shape[0] == len(table)}")
print(f"empty histories     : {(seq['lengths'] == 0).mean():.1%} of rows (first pitch of a PA)")

**How to read it.** The token width `d_tok` is what one "word" of the at-bat sentence is: two
16-dim embeddings (the family and the outcome of a prior pitch) plus the seven physical channels. The
`row_id` alignment is the contract that lets the raw tensors be scored through the *same* harness as
the tabular models — no view re-implements evaluation (SPEC §8). The empty-history fraction is the
first pitch of every PA, where only the static context (and, for OM, matchup memory) can matter.

In [ ]:
# Figure: (a) sequence-length distribution on the eval rows; (b) per-channel scale summary.
eval_mask = masks["val"]
L = seq["lengths"][eval_mask]
chan = np.stack([seq[c] for c in SEQUENCE_FLOAT_CHANNELS], axis=-1)   # (n, max_len, C)
valid = seq["mask"].astype(bool)
flat = chan[valid] if valid.any() else np.zeros((1, len(SEQUENCE_FLOAT_CHANNELS)))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.5, 4.2))
style_axes(axL); style_axes(axR)

axL.hist(L, bins=range(0, MAX_LEN + 2), color=VIEW_COLORS["O"], alpha=0.85, align="left")
axL.set_xlabel("ordered history length (prior pitches this PA)")
axL.set_ylabel("eval rows")
axL.set_title("Sequence-length distribution\n(what the GRU actually reads per decision)")

means = flat.mean(axis=0)
stds = flat.std(axis=0)
ypos = np.arange(len(SEQUENCE_FLOAT_CHANNELS))
axR.barh(ypos, means, xerr=stds, color=VIEW_COLORS["L1"], alpha=0.85,
         error_kw=dict(ecolor=REF_COLOR, lw=1))
axR.set_yticks(ypos); axR.set_yticklabels(list(SEQUENCE_FLOAT_CHANNELS))
axR.axvline(0.0, color=REF_COLOR, lw=0.8)
axR.set_xlabel("mean +/- sd over valid (masked) history positions")
axR.set_title("Float-channel scales (raw)\ndvelo/dloc are the ordered transitions")
plt.show()

*Caption.* Left: most ordered histories are short (a plate appearance is a handful of pitches),
which bounds how much *order* there can be to find — the SPEC §13 expectation that order barely helps
starts here. Right: the raw channels live on wildly different scales (velocity `~90`, plate coords
`~1–3`, `dvelo` `~0–10`), so `FeatureEncoders` standardises each channel over the valid training
positions before the GRU sees it — the recurrence trains far more stably standardised. The two
**transition** channels `dvelo`/`dloc` are the ordered raw material for the tunneling/velocity-jump
mechanisms the positive world plants.

## 3. The five capacity-matched architectures (the design exhibit)

One `_SeqNet` per (view, target), sharing embedding width `16`, hidden width `64`, a single recurrent
layer, dropout `0.1`, and the same MLP head. The five views differ **only** in the sequence branch —
this is the ladder that varies *information*, not *capacity* (D45):

| view | sequence branch | what it can see | order? |
|---|---|---|---|
| **C**  | none (static MLP) | context only | — |
| **U**  | masked mean-pool of token projections | the *bag* of prior pitches | **no, by construction** |
| **L1** | GRU fed the **last** token only | context + last pitch | previous-pitch only |
| **O**  | the **same** GRU over the **full ordered** sequence | context + full ordered history | **yes** |
| **OM** | O + a matchup-memory branch | O + batter-vs-pitcher memory | yes + cross-PA |

`U` is order-invariant because a masked mean is symmetric in its inputs (`THEORY.md` §2). `L1` and `O`
are the **identical network** — same embeddings, same GRU, same head — differing only in the forward
path (`_last_token` vs `_full_seq`), so their parameter counts are equal *by construction*
(`THEORY.md` §§3–4). For the outcome target the current-action embedding is concatenated to the head
(D22) — SPEC §0's "after conditioning on the current pitch." The cell below builds all five and reads
off the parameter counts.

In [ ]:
# Build one _SeqNet per view (selection target) at the shared width; read off parameter counts.
enc_arch = FeatureEncoders(max_len=MAX_LEN).fit(table.loc[masks["train"]], fit_match=True)
_b = enc_arch.make(table.loc[masks["train"]], "OM", "selection")
static_dim, match_dim = int(_b["static"].shape[1]), int(_b["matchup"].shape[1])

param_rows = []
for v in VIEWS:
    seed_everything(SEED)
    net = _SeqNet(v, "selection", static_dim=static_dim, match_dim=match_dim,
                  embed_dim=EMBED, hidden=HIDDEN, num_layers=1, dropout=0.1)
    param_rows.append((v, int(sum(p.numel() for p in net.parameters()))))

live = dict(param_rows)
print(f"{'view':<4} {'params (live)':>14} {'params (committed)':>20}   note")
for v, n in param_rows:
    note = ""
    if v == "O":
        note = "== L1 (the capacity match: identical net)" if live.get("L1") == n else "(expected == L1)"
    elif v == "L1":
        note = "== O below (identical net; only the fed information differs)"
    print(f"{v:<4} {n:>14,} {COMMITTED_PARETO[v][0]:>20,}   {note}")
print()
print(f"capacity match holds live: n_params(L1) == n_params(O)  ->  {live.get('L1') == live.get('O')}")
print("(live counts differ from the committed demo because static_dim depends on the world's "
      "categorical vocab; the L1 == O invariant holds regardless.)")

# The Okabe-Ito ladder-of-widths figure (committed-validation parameter counts).
fig, ax = new_fig((7.6, 4.0))
xs = np.arange(len(VIEWS))
ax.bar(xs, [COMMITTED_PARETO[v][0] for v in VIEWS], color=[VIEW_COLORS[v] for v in VIEWS], alpha=0.9)
ax.set_xticks(xs); ax.set_xticklabels(VIEWS)
ax.set_ylabel("parameters (committed validation)")
ax.set_title("Capacity-matched ladder: params reported, L1 == O forced equal\n"
             "(a bigger O would confound information with capacity -- D45)")
for i, v in enumerate(VIEWS):
    ax.text(i, COMMITTED_PARETO[v][0], f"{COMMITTED_PARETO[v][0]:,}", ha="center", va="bottom", fontsize=9)
plt.show()

*Caption + why matching matters.* The parameter counts rise `C < U < L1 = O < OM`, and the one
comparison that carries the order claim — **L1 vs O** — is parameter-*identical* by construction
(`31,112` each on the committed run). That is the whole design: if we let `O` be a bigger network and
it beat `L1`, we could not tell whether *order* or *size* won. By forcing `L1` and `O` to be the same
net with only the fed information differing, a measured `O`-over-`L1` gain is ordered information, full
stop (`THEORY.md` §4). `C` and `U` are matched on *width* but are necessarily different function
classes (non-recurrent), so `U − O` is read cautiously and the capacity-matched twin `L1 − O` carries
the headline.

## 4. Training discipline — early stopping, seeds, device-agnostic code, the honest compute table

Each model is fit with **Adam** (`lr=3e-3`, `weight_decay=1e-4`, `batch_size=512`) with **early
stopping on the validation fold** (`patience=8`, restoring best-val weights); the falsification refits,
which have no external val fold, hold out a seeded `internal_val_frac=0.2` slice so the ordered `O`
view still stops before overfitting and generalises as well as its twin `L1` (`THEORY.md` §7). The
compact hyperparameters (embed `16`, hidden `64`, `≤2` layers) are the predeclared **D46** defaults,
identical across views — the equal-budget discipline of SPEC §7, so the only thing that varies is the
information each encoder reads. Code is **device-agnostic** (`resolve_device`, `--device auto`): the
same code runs on a desktop CPU and a Colab T4.

In [ ]:
# The honest compute table (SPEC §7 Pareto discipline) -- committed validation, null world, 6 epochs.
print("COMMITTED-VALIDATION PARETO ROW (null world, 6 epochs, ~1.0 GB peak; selection target):")
print(f"  {'view':<4} {'params':>9} {'seconds':>9}   gain-per-second axis")
for v in VIEWS:
    p, s = COMMITTED_PARETO[v]
    tag = ""
    if v == "O":
        dt = COMMITTED_PARETO["O"][1] - COMMITTED_PARETO["L1"][1]
        tag = f"O premium over L1 = +{dt:.1f}s at EQUAL params (pure wall-clock)"
    print(f"  {v:<4} {p:>9,} {s:>9.1f}   {tag}")
print()
print("Reading (THEORY §8): L1 and O have identical params, so the only cost separating them is TIME.")
print("If Delta_order is small (SPEC §13's expectation), O is Pareto-dominated by L1 -> the cheap")
print("model wins the frontier. Whether the whole deep rung earns its ~orders-more compute vs WS1-WS3")
print("is the SPEC §7 question the final Pareto plot answers.")

**CPU vs. Colab — pick one path (decision D46).** WS6 is the study's only GPU-relevant step, and
only the recurrent O/OM fits benefit.

- **RUNBOOK WS6.1 — local CPU.** `pip install -e ".[deep]"`, `--device auto` → CPU. Honest and simple;
  the full O-view GRU over `~3.85M` sequences is the slowest step in the study (estimate: several
  hours). Checkpoints per `(view, target)`, so it is fully resumable.
- **RUNBOOK WS6.2 — free Colab T4.** The self-contained `workstreams/ws6_deep_seq/colab_ws6.ipynb`
  runs the **same** `pitchseq`/WS6 code on a free Colab T4 GPU — `~10–20×` faster for the recurrent
  fits (`~1` hour end-to-end), the **recommended** route.

> **AMD GPU caveat (ROCm / DirectML).** Sean's desktop GPU is AMD. PyTorch's Windows wheels are
> CPU/CUDA only — there is no stable Windows ROCm build, and `torch-directml` is an unofficial,
> often-lagging backend. **Do not fight the AMD GPU**: use WS6.1 (CPU) for correctness or WS6.2 (free
> Colab T4) for speed. Only WS6's *optional* Transformer demo is genuinely GPU-preferred.

## 5. Evaluation through the shared harness — per-view losses and both Δs

One call to `run_ws6` drives the whole pipeline: it fits all five capacity-matched views on both
targets (early-stopping on val), writes standard-schema predictions, scores them **only** through the
shared harness (`compare_views` → `Δ_order` / `Δ_matchup` with pitcher-game clustered CIs), and — on
synthetic worlds — runs the D47 falsification battery and the motif probe. Every downstream exhibit
(§§6–9) reads the returned report. The cross-workstream WS2/WS3 comparison is a **read-off**: paste
their O-view losses into the comparison line (WS6 refits nothing from other workstreams).

In [ ]:
def run_pipeline(mode):
    """Drive run_ws6 once (all five views x both targets, harness scoring, D47 on synth worlds)."""
    return run_ws6(
        source=str(REAL_TABLE_PATH) if mode == "real" else None,
        synth=_SYNTH[mode], out="results/ws6_nb", views=VIEWS, targets=TARGETS_,
        n_games=NB_N_GAMES, seed=WORLD_SEED, epochs=NB_EPOCHS,
        hidden=HIDDEN, embed=EMBED, batch=BATCH, max_len=MAX_LEN, device="auto",
        n_boot=NB_N_BOOT, n_perm=NB_N_PERM, ci_boot=NB_CI_BOOT,
        write_outputs=False, config=CONFIG,
    )


RES = run_pipeline(DATA_MODE)
print(_format_headline(RES))

**The reading rules (binding).**

$$\Delta_{\text{order}} = \text{Loss}(\min[U, L1]) - \text{Loss}(O), \qquad
  \Delta_{\text{matchup}} = \text{Loss}(O) - \text{Loss}(OM)$$

`Δ_order` is **negatively biased under the null** (the min of two noisy losses is optimistic, decision
D21): the criterion is *significantly positive* (clustered CI lower bound `> 0`), and a small
**negative** reads as *consistent with no ordering effect*, never "order hurts." `Δ_matchup` takes
**no minimum**, so it carries no such bias and is read directly — a significant negative is a *real*
out-of-sample cost (`THEORY.md` §5).

In [ ]:
# The central table (live) + the D21-annotated deltas + the WS2/WS3 read-off line.
sc = RES["scores"]
bl = RES.get("baseline_selection_loss", {})

def _f(x, spec="+.4f"):
    try:
        return format(float(x), spec)
    except (TypeError, ValueError):
        return "  n/a "

print(f"{'view':<4} {'params':>8} {'sel_ll':>9} {'out1_ll':>9}")
for v in VIEWS:
    p = RES["pareto"]["selection"].get(v, {}).get("n_params", 0)
    s = sc["selection"]["central"].get(v, {}).get("log_loss")
    o = sc["outcome1"]["central"].get(v, {}).get("log_loss")
    print(f"{v:<4} {p:>8,} {_f(s, '.4f'):>9} {_f(o, '.4f'):>9}")
if bl:
    print("selection references : " + "  ".join(f"{k}={_f(x, '.4f')}" for k, x in bl.items()))
print("WS2/WS3 read-off (Phase 2): learned O sel vs WS3 GBDT-O {PLACEHOLDER} / WS2 grammar-O {PLACEHOLDER};")
print("                           learned O out1 vs WS3 GBDT-O {PLACEHOLDER}   (the R-axis, §9)")

print("\nDelta_order (min[U,L1] - O), clustered CI, read under D21:")
for t in TARGETS_:
    vd = sc[t].get("val_deltas", {})
    do, ci = vd.get("deltas", {}).get("delta_order"), vd.get("delta_order_ci")
    ann = ""
    if ci is not None:
        ann = "significantly positive (order helps)" if ci.get("lo", 0) > 0 else \
              "not significantly positive: consistent with no ordering effect (D21)"
    lo = _f(ci["lo"]) if ci else " n/a "; hi = _f(ci["hi"]) if ci else " n/a "
    print(f"  {t:<10}: {_f(do)}  CI[{lo}, {hi}]   {ann}")

print("\nDelta_matchup (O - OM), clustered CI, read DIRECTLY (no min-bias):")
for t in TARGETS_:
    vd = sc[t].get("val_deltas", {})
    dm, ci = vd.get("deltas", {}).get("delta_matchup"), vd.get("delta_matchup_ci")
    lo = _f(ci["lo"]) if ci else " n/a "; hi = _f(ci["hi"]) if ci else " n/a "
    neg = "  <- significantly negative: real matchup-feature cost (M-)" if (ci and ci.get("hi", 0) < 0) else ""
    print(f"  {t:<10}: {_f(dm)}  CI[{lo}, {hi}]{neg}")

**The three-model-family Δ_matchup pattern.** On the synthetic worlds — where **no** matchup
effect is planted — `Δ_matchup` comes out significantly **negative** (committed validation: `−0.0036`,
CIs excluding 0, on both targets). Because `Δ_matchup` has no `min`-bias, this is a genuine
out-of-sample **fragmentation cost**: the matchup branch adds estimation variance without signal, so
the model generalises *worse* with it. WS6 is now the **third** model family to measure this — after
**WS3** (GBDT, both worlds) and **WS5** (tabular MDP, the D42/greedy-optimism exhibits). Three
independent function classes (trees, a tabular MDP, a recurrent net) agreeing turns a single-workstream
curiosity into a robust methodological statement: *more features is not more information.* On real data
a negative `Δ_matchup` reads as "no matchup signal a learned encoder can see **and** a real cost of
carrying the sparse features" — checked with the support diagnostics — the pre-written `M−` branch of
§9.

## 6. The targeting lesson — why the aggregate was null but the repeat-context edge was real

The null world's planted habit (no-three-in-a-row) **only acts when the two most-recent prior pitches
are the same family** — the "repeat context." So the ordered edge the GRU can find lives on a *subset*
of rows, and the aggregate `Δ_order` mixes those signal rows with a majority of noise-only rows. This
is the general lesson, made live. With per-row effect `δ` present on a subset of support fraction `ρ`
and absent elsewhere,

$$\underbrace{\text{aggregate effect}}_{\rho\,\delta} \quad\text{vs.}\quad \underbrace{\text{subset effect}}_{\delta},
\qquad \frac{z_{\text{sub}}}{z_{\text{agg}}} = \frac{1}{\sqrt{\rho}} > 1,$$

so the aggregate dilutes the point estimate by `ρ` and the power by `√ρ` — the effect must be
evaluated on the subset it acts on (`THEORY.md` §5). The harness exposes exactly this via its slice
machinery — here, the `row_mask` of `_l1_twin_delta_ci` restricted to the repeat-context rows
(`_repeat_context_mask`).

In [ ]:
# Aggregate selection Delta_order vs the repeat-context capacity-matched twin delta (L1 - O).
fal = RES.get("falsification")
sel_vd = sc["selection"].get("val_deltas", {})
agg_do, agg_ci = sel_vd.get("deltas", {}).get("delta_order"), sel_vd.get("delta_order_ci")

print("THE TARGETING LESSON (selection target)")
print("-" * 66)
lo = _f(agg_ci["lo"]) if agg_ci else " n/a "; hi = _f(agg_ci["hi"]) if agg_ci else " n/a "
print(f"AGGREGATE  Delta_order (all eval rows) : {_f(agg_do)}  CI[{lo}, {hi}]")

rc = _repeat_context_mask(table)
print(f"repeat-context rows (habit active)     : {int(rc.sum()):,} of {len(table):,}  "
      f"(support fraction rho = {rc.mean():.3f})")

if fal is not None and fal.get("world") == "null":
    gci = fal.get("grammar_repeat_ci", {})
    print(f"SUBSET     twin delta L1 - O | repeat   : {_f(fal.get('grammar_repeat_delta'))}  "
          f"CI[{_f(gci.get('lo'))}, {_f(gci.get('hi'))}]   (n_repeat_rows={fal.get('grammar_repeat_n')})")
    print(f"grammar verdict                        : {fal.get('grammar_verdict')}")
    print("\n-> Same habit: DILUTED to n.s. in the aggregate, CI-POSITIVE on the subset it acts on.")
else:
    print("(set DATA_MODE='synth_null' to see the repeat-context twin delta; the D47 grammar gate)")

# The committed-validation numbers (state as done).
print("\nCOMMITTED VALIDATION (null world, 6 epochs):")
print("  AGGREGATE selection Delta_order : +0.0006  (not significant -- diluted)")
print("  SUBSET twin delta | repeat      : +0.0115  CI[+0.0064, +0.0161]  (n_repeat_rows=1136)")
print("  power gain from targeting (THEORY §5): factor 1/sqrt(rho) > 1")

*Caption.* The exhibit in one line: **the same effect is hidden in the aggregate and detected on
the subset it acts on.** The aggregate selection `Δ_order` is null (`+0.0006`, committed) because the
habit does nothing on the `~(1−ρ)` of rows that are not repeat contexts, and their noise drowns the
signal; the repeat-context twin delta is CI-positive (`+0.0115`) because it grades the model exactly
where the rule lives. This binds on the real-data reading (the H-axis, §9): a genuine ordered effect
might live only in `long_pa` / `two_strike` and be invisible in the headline `Δ_order` — so **read the
slice deltas, not just the total.** The `√(1/ρ)` power gain (`THEORY.md` §5) is why subset-targeted
evaluation wins.

## 7. The opacity exhibit — the GRU uses the grammar its probabilities won't cleanly expose

Section 6 established (via the loss) that the ordered GRU *uses* the grammar. Does its explicit output
distribution *expose* the rule the way WS2's inspectable motif table does? The gradient-free
`motif_repeat_probe` asks, per family `X`: is a third consecutive `X` made less likely after a repeat?
It compares two synthetic length-2 histories that **end in the same token `X`** and differ only in the
second-to-last pitch — `[X, X]` vs `[Y, X]` averaged over `Y ≠ X` — holding context at the training
mean and channels in-distribution, so an `L1` model would predict identically and only the ordered `O`
can react (`_probe_next`). `suppression(X) = P(X | [Y,X]) − P(X | [X,X])`, positive when the habit is
exposed.

In [ ]:
# The motif-rediscovery probe: P(same family 3rd) after [X,X] vs [Y,X], per family.
mot = RES.get("motif")
if mot is None:
    print("(motif probe runs on synthetic worlds with both targets; set DATA_MODE='synth_null')")
else:
    print(f"{'fam':<4} {'P(X|XX)':>9} {'P(X|YX)':>9} {'suppression':>12}")
    for r in mot["per_family"]:
        print(f"{r['family']:<4} {_f(r['p_same_after_repeat'], '.4f'):>9} "
              f"{_f(r['p_same_after_mixed'], '.4f'):>9} {_f(r['suppression']):>12}")
    print("-" * 40)
    print(f"mean_suppression = {_f(mot['mean_suppression'])}   "
          f"frac_suppressed = {_f(mot['frac_suppressed'], '.2f')}   "
          f"motif_present = {mot['motif_present']}")

    fig, ax = new_fig((8.2, 4.2))
    fams = [r["family"] for r in mot["per_family"]]
    supp = [r["suppression"] for r in mot["per_family"]]
    colors = [OKABE_ITO["bluish_green"] if s > 0 else OKABE_ITO["vermillion"] for s in supp]
    ax.bar(np.arange(len(fams)), supp, color=colors, alpha=0.9)
    ax.axhline(0.0, color=REF_COLOR, lw=1)
    ax.set_xticks(np.arange(len(fams))); ax.set_xticklabels(fams)
    ax.set_ylabel("suppression = P(X|YX) - P(X|XX)")
    ax.set_title("Motif probe: does the O GRU expose no-three-in-a-row?\n"
                 "green = suppressed (rule shows); red = anti-suppressed (rule hidden)")
    plt.show()

print("\nCOMMITTED VALIDATION (null world, 6 epochs): motif_present = False")
print("  mean_suppression = -0.0063 ; frac_suppressed = 0.43 (< 0.5)")
print("  FF: +0.05 (suppressed)  BUT  FC: -0.057 (anti-suppressed)")

**The opacity reading — the deep-learning bargain in one exhibit.** The GRU's **loss** detects
the grammar (the repeat-context twin delta is CI-positive, §6) while its explicit per-family
**probabilities** do **not** cleanly expose it (`motif_present = False`; committed mean suppression
`−0.0063`, only `43%` of families suppressed — `FF +0.05` but `FC −0.057`). This is not a contradiction
(`THEORY.md` §6): the loss is a *frequency-weighted* average over the realized next pitches, while the
probe is an *unweighted* per-family counterfactual level on partly out-of-support inputs (a pitcher
rarely repeats a cutter, so `[FC,FC]` is near the edge of support and the readout is noisy). The
contrast with **WS2** is the point: WS2's variable-order grammar stores the motif as an **explicit,
inspectable table** — you can read "after `[X,X]`, `P(X)` drops" straight off it, all top motifs
correctly signed — whereas WS6's GRU folds the same rule into 64 continuous hidden units and will not
say it cleanly. *The net uses the pattern (loss) without exposing it (probabilities).* For
interpretability claims this is decisive: **an accuracy claim about a learned model does not license a
mechanism claim** — interpretability is a property to measure, not assume. What more training / full
data would change is a Phase-2 question (`{PLACEHOLDER}` for the real-data rerun).

## 8. Falsification — the D47 battery as completed validation

WS6 reruns the SPEC §11 oracle as a real-model test through the D17 factory adapter. On the **null**
world the two D47 verdicts must both fire: `GRU_GRAMMAR_DETECTED` (the ordered selection GRU beats its
capacity-matched twin `L1` on the repeat-context subset) **and** `GRU_OUTCOME_QUIET` (no ordered
*outcome* dependence — `Δ_order` not significantly positive and the token-order permutation test does
not fire, read under D21). The positive world is the outcome-mechanism recovery control.

In [ ]:
# Live D47 battery from the pipeline report (synthetic worlds).
fal = RES.get("falsification")
if fal is None:
    print("(real data: the D47 permutation battery is synthetic-only -- infeasible at ~3.85M rows.")
    print(" set DATA_MODE='synth_null' or 'synth_positive' to run it; on real data the central")
    print(" table carries the ablation and the synthetic worlds carry the recovery/permutation checks.)")
else:
    perm = fal.get("permutation", {})
    print(f"world : {fal.get('world')}")
    if fal.get("world") == "null":
        gci = fal.get("grammar_repeat_ci", {})
        oci = fal.get("outcome_delta_order_ci", {})
        print(f"  GRAMMAR : {fal.get('grammar_verdict')}  repeat-context L1-O = "
              f"{_f(fal.get('grammar_repeat_delta'))} CI[{_f(gci.get('lo'))},{_f(gci.get('hi'))}] "
              f"(n_repeat={fal.get('grammar_repeat_n')})")
        print(f"  OUTCOME : {fal.get('outcome_verdict')}  Delta_order="
              f"{_f(fal.get('outcome_delta_order'))} CI[{_f(oci.get('lo'))},{_f(oci.get('hi'))}]  "
              f"perm p={_f(perm.get('p_value'), '.3f')} fired={perm.get('fired')}")
    else:
        print(f"  outcome Delta_order = {_f(fal.get('outcome_delta_order'))}  "
              f"perm p={_f(perm.get('p_value'), '.3f')} fired={perm.get('fired')}  "
              f"detected={fal.get('order_effect_detected')}")
        print(f"  recovered whiff-lift={_f(fal.get('recovered_whiff_lift'))}  "
              f"planted={_f(fal.get('planted_whiff_lift'))}  "
              f"ratio={_f(fal.get('recovery_ratio'), '.3f')}  sign_ok={fal.get('recovered_sign_ok')}")
    print(f"  D47 verdict : {fal.get('d47_verdict')}   (pass={fal.get('d47_pass')})")

**Completed synthetic validation (state these as done — the committed WS6a run).**

- **Null world — D47 PASS.**
  - `GRU_GRAMMAR_DETECTED`: repeat-context twin delta `L1 − O = +0.0115`, CI `[+0.0064, +0.0161]`,
    `n_repeat_rows = 1136` (the aggregate selection `Δ_order` is `+0.0006`, n.s. — the targeting lesson,
    §6).
  - `GRU_OUTCOME_QUIET`: outcome `Δ_order = −0.0004`, CI `[−0.0023, +0.0011]`; permutation `p = 0.091`
    (not fired); locked test `−0.0011` (n.s.) — the correct null read under D21.
  - `Δ_matchup` significantly negative on both targets (`−0.0036`) — the three-family cost pattern (§5).
- **Opacity exhibit:** `motif_present = False` at demo scale (§7) — the loss uses the grammar the
  probabilities won't expose.
- **Positive world — direction-level coverage.** The lean pipeline test asserts the *direction* at
  small scale: the outcome `O` model's predicted whiff lift on triggered vs untriggered rows is positive
  (`recovered_sign_ok`), the honest first-class verdict **`GRU_MECHANISM_DIRECTIONAL`** (mirroring
  WS4/WS5). **What full-scale Phase 2 must confirm:** the full **`GRU_MECHANISM_RECOVERED`** verdict —
  the aggregate outcome `Δ_order` CI above 0 **and** the permutation test firing — reported with a
  **recovery ratio** like WS3's `0.955` (`{PLACEHOLDER}`). The `dvelo` channel carries the mechanism
  directly, so the GRU should recover it once the sample is large enough to certify the magnitude.

## 9. Results — branched interpretation (Representation × Order × Matchup)

The real-data result is read on **three independent axes**. The **representation** axis (R+/R=/R−) is
WS6's actual question — does the *learned* `O` beat WS3's *engineered* `O`? The **order** axis
(H1/H2/H3) is the within-WS6 `Δ_order`, read under D21 **and with the targeting lesson applied** (check
the repeat-context / slice deltas, §6). The **matchup** axis (M+/M0/M−) is `Δ_matchup`, read directly
and in the three-family context (§5). The selector below computes each axis from the live report; the
representation axis needs WS3's O-loss pasted in (a read-off), so fill `WS3_O_OUTCOME1_LL` to resolve
it. The write-ups that follow stand alone once the numbers are filled.

In [ ]:
# --- Branch selector (outcome-1 target) ---------------------------------------------
# Representation axis: paste WS3's GBDT-O outcome-1 log loss from results/ws3/ws3_report_real.json.
WS3_O_OUTCOME1_LL = None   # <-- fill from WS3's report (a read-off); None until pasted
R_MATERIAL = 2e-3          # log-loss (nats) a learned edge must clear to count as beating WS3

o1_vd = sc["outcome1"].get("val_deltas", {})
o1_loss = {v: sc["outcome1"]["central"].get(v, {}).get("log_loss") for v in VIEWS}

# Order axis (H1/H2/H3), read under D21.
do, do_ci = o1_vd.get("deltas", {}).get("delta_order"), o1_vd.get("delta_order_ci")
best_hist_gain = max((o1_loss["C"] - o1_loss[v]) for v in ("U", "L1", "O")
                     if o1_loss["C"] is not None and o1_loss[v] is not None) \
                 if o1_loss.get("C") is not None else None
SEL_MATERIAL = 3e-3
if best_hist_gain is None:
    order_branch = "n/a"
elif best_hist_gain <= SEL_MATERIAL:
    order_branch = "H3"                                   # nothing beats C
elif do_ci is not None and do_ci.get("lo", 0) > 0:
    order_branch = "H1"                                   # O beats min[U,L1] by more than the CI
else:
    order_branch = "H2"                                   # history beats C, order ~ U/L1

# Matchup axis (M+/M0/M-), read directly.
dm, dm_ci = o1_vd.get("deltas", {}).get("delta_matchup"), o1_vd.get("delta_matchup_ci")
if dm_ci is None or dm is None:
    matchup_branch = "M0"
elif dm_ci.get("lo", 0) > 0:
    matchup_branch = "M+"
elif dm_ci.get("hi", 0) < 0:
    matchup_branch = "M-"
else:
    matchup_branch = "M0"

# Representation axis (R+/R=/R-): learned O vs engineered WS3-O.
if WS3_O_OUTCOME1_LL is None or o1_loss.get("O") is None:
    repr_branch = "R? (fill WS3_O_OUTCOME1_LL to resolve)"
else:
    gap = WS3_O_OUTCOME1_LL - o1_loss["O"]               # >0: learned O beats engineered O
    repr_branch = "R+" if gap > R_MATERIAL else ("R-" if gap < -R_MATERIAL else "R=")

print("=" * 70)
print(" WS6 RESULTS -- branch selector (outcome-1 target)")
print("=" * 70)
print(f" world / mode          : {RES.get('world', DATA_MODE)}")
print(f" outcome-1 log loss    : " + "  ".join(f"{v}={_f(o1_loss[v], '.4f')}" for v in VIEWS))
print(f" learned O vs WS3-O    : WS3_O={WS3_O_OUTCOME1_LL}  (read-off; None until pasted)")
print(f" Delta_order (outcome) : {_f(do)}  CI[{_f(do_ci['lo']) if do_ci else 'n/a'}, "
      f"{_f(do_ci['hi']) if do_ci else 'n/a'}]   (also check the repeat-context/slice twin, §6)")
dm_txt = (f"{_f(dm)}  CI[{_f(dm_ci['lo'])}, {_f(dm_ci['hi'])}]" if dm is not None and dm_ci else "n/a")
print(f" Delta_matchup (out)   : {dm_txt}")
print("-" * 70)
print(f" REPRESENTATION BRANCH : {repr_branch}")
print(f" ORDER BRANCH          : {order_branch}")
print(f" MATCHUP BRANCH        : {matchup_branch}")
print("=" * 70)
print(" -> read the matching branch write-ups in the markdown below.")

### Representation axis (learned `O` vs engineered WS3-`O`) — R+ / R= / R−

**R+ — the learned representation beats the engineered features.** GRU-`O` posts a lower loss than
WS3's GBDT-`O` (and WS2's grammar): the learned encoder found ordered signal the hand-designed slots
and difference features (D11) missed. The strong deep-learning result — but *before believing it*,
inspect **what** it found: run the motif probe and the per-slice deltas on the real-data model, and
confirm the edge concentrates where order should matter (`long_pa`, `two_strike`), not as an artifact
of the GRU's extra smoothing of continuous physics. If it survives, WS6 justifies going beyond
engineered features and the cross-workstream paper leads with it.

**R= — the learned representation ties the engineered features.** GRU-`O ≈` WS3-`O` within CIs: the
engineered encoding already captured the available ordered signal. This is a **Pareto** statement as
much as a loss one (`THEORY.md` §8) — if the two tie on loss, the far cheaper GBDT wins the frontier,
and the honest headline is "engineered features suffice; the compute peak did not buy new signal." The
most likely outcome (SPEC §13) and a clean one: the deep model *confirms* the engineered result.

**R− — the learned representation underperforms.** GRU-`O` posts a *higher* loss than WS3-`O`. At this
scale this is an **optimisation or data limit**, not "learning can't help": the recurrent fit
under-trained, the budget too small, the sequences too short to reward a recurrence over a tree. **Read
the training curves first** (did val loss still fall at the epoch cap?), check the capacity-matched twin
`L1 − O` (a negative twin at equal parameters points at optimisation, `THEORY.md` §4), and report it
honestly as a statement about *this* comparison, not about deep learning.

### Order axis (within WS6, outcome-1 `Δ_order`, D21 + the targeting lesson) — H1 / H2 / H3

**H1 — `O` beats both `U` and `L1` (CI lower bound > 0): genuine ordered dependence.** Ordered
information the learned encoder captured. Because of the targeting lesson (§6), a real effect may be
**stronger on the subset it acts on than in the aggregate** — check the repeat-context / per-slice twin
deltas, not only the headline `Δ_order`, and read the locked-test row. On the outcome target this is
the live finding-#2 signal the prescriptive workstreams try to use.

**H2 — `O ≈ U/L1` but both beat `C`: history matters, order does not.** History lowers loss below
context-only, but the fully ordered `O` does not refine on the better of `U`/`L1` (`Δ_order` not
significantly positive, *and* the repeat-context twin not positive either — the targeting check that
keeps H2 honest). The SPEC §13 expectation ("O barely beats L1"), reported without embarrassment.

**H3 — nothing beats `C`: no sequencing signal at all.** No history view lowers loss below
context-only — the cleanest finding-#2 null for the learned model. Does not deny finding #1 (selection
structure may still exist — the null world shows exactly that separation). Audit calibration and the
training curves before accepting, since an under-optimised recurrence can masquerade as H3.

### Matchup axis (outcome-1 `Δ_matchup`, read directly, three-family context) — M+ / M0 / M−

**M+ — `OM` beats `O` (CI lower bound > 0): real batter–pitcher adaptation.** Longer-term adaptation
the learned cross-PA representation captured, which the `first_pitch` slice (where matchup memory is
`OM`'s only added information) must corroborate. It would be the study's *first* positive matchup signal
— WS3 and WS5 both found the block *costs*.

**M0 — `OM ≈ O`: no detectable matchup memory.** CI spans 0. Expected if within-game/season rematches
are too thin for even a pooled encoder to resolve.

**M− — `OM` significantly below `O`: matchup features cost out-of-sample.** No `min`-bias, so a *real*
cost — the same fragmentation reading as WS3 (`../ws3_gbdt_stack/THEORY.md` §6.3). **Observed on both
synthetic WS6 targets** (`−0.0036`) and now a **three-family** pattern (WS3, WS5, WS6). Read as "no
matchup signal a learned encoder can see **and** a real fragmentation cost," checked with the support
diagnostics — that three independent model families agree makes it a robust statement about
sparse-feature fragmentation, not about baseball.

### Cross-reading the grid

The honest headline is a triple `(R, H, M)`. The **most anticipated** cell is **R= × H2 × M−**: the
learned representation confirms the engineered result (order barely helps → the cheap model wins the
frontier) and the matchup block costs — a modest, defensible finding with a clean compute-vs-gain
reading and a three-family methodological note. The **strongest** cell is **R+ × H1 × M+**: the learned
model finds ordered *and* matchup signal the engineered features missed — a live signal for the whole
prescriptive phase, believed only after the probes confirm *what* it found. The **cleanest
confirmation-of-null** is **R= × H3 × M0**. Whichever triple fires, it is read under the firewall (§10):
a predictor measures *prediction*, and only the OPE/RL workstreams can test whether any of it is
*prescriptive*.

## 10. Discussion and limitations

**The representation question, honestly.** WS6 exists to ask whether a learned encoding beats
engineered features and the explicit grammar. Under **R+** it does, and the burden shifts to
*interpreting* the gain — which is exactly why the opacity exhibit matters: "the GRU won" is not
self-explanatory, and a learned edge must be probed before it is believed. Under **R=** (the SPEC §13
expectation) the durable contribution is a **Pareto** one — the deep model confirms WS3's engineered
`O` already captured the signal, so the compute peak bought no new prediction. Under **R−** the honest
content is a scale statement read from the training curves, never inflated into "deep learning fails."

**The opacity trade.** The sharpest thing WS6 shows is a *dissociation*: the GRU's loss demonstrably
**uses** the planted grammar while its explicit probabilities do **not** cleanly expose it
(`motif_present = False`). WS2 says the rule out loud; WS6 uses the same rule but will not — the
deep-learning bargain (more power to learn, less ability to explain). For the cross-workstream paper
this is the concrete case that **interpretability is measured, not assumed**.

**Capacity matching's limits.** Matched parameters (`L1 = O`) do **not** match inductive bias (`C`/`U`
are non-recurrent function classes) or the optimisation landscape (`O` has the harder fit), so `L1 − O`
is the clean order comparison and a null-or-negative twin is consistent with "no order information" *or*
"`O`'s harder optimisation swallowed a small effect at this scale" (`THEORY.md` §4). Demo-scale honesty:
the completed validation is a 6-epoch synthetic run; the motif per-family readout and the positive-world
*magnitude* recovery sharpen with scale, and the positive-world verdict at demo scale is honestly
`GRU_MECHANISM_DIRECTIONAL`.

**Compute-vs-gain (SPEC §7).** WS6 is the study's compute peak on the predictive side; `L1` and `O`
have identical parameters, so the `O` premium is pure wall-clock (`~1.8×`, `THEORY.md` §8). If order
barely helps, `O` is Pareto-dominated by `L1` — the cheap model wins the frontier, and whether the whole
deep rung earns its orders-more compute against WS1–WS3 is the SPEC §7 question the final Pareto plot
answers.

**The Transformer demo is optional and out of scope** (SPEC §12.4): a demonstration for the long
cross-PA test, kept out of the acceptance gates and exercised by a single smoke test — no Transformer
result is a WS6 finding.

**The firewall.** WS6 measures whether ordered history *predicts* selection and outcomes out-of-sample
(findings #1/#2); it does **not** establish that *changing* the sequence would change outcomes
(finding #3). `pitch_type` is a classifier output and the action is at family granularity (SPEC §1, §4);
falsification is synthetic-only (D47); there are no catcher/umpire effects and a single reward metric.
Turning finding #2 into finding #3 is the OPE/RL workstreams' job, tested rather than assumed
(`../ws3_gbdt_stack/THEORY.md` §8).

## 11. Reproducibility appendix

**Phase-2 runs go through the CLI, not this notebook** (RUNBOOK Step WS6). WS6 needs only the decision
table (Step 1); it builds its own sequence tensors and refits nothing from other workstreams.

```powershell
conda activate statcast; cd ~\pitch-sequencing-research
pip install -e ".[deep]"                      # the only step that needs PyTorch (D46)

# WS6.1 -- LOCAL CPU. Calibrate one view/3 epochs first, then the resumable full ladder:
python workstreams/ws6_deep_seq/run_ws6.py --table data/processed/decision_table.parquet `
    --views O --targets selection --epochs 3 --device auto --out results/ws6_calib/
python workstreams/ws6_deep_seq/run_ws6.py --table data/processed/decision_table.parquet `
    --device auto --epochs 30 --out results/ws6/

# Synthetic D47 gates (no data; WS6 builds the world itself; ~minutes each):
python workstreams/ws6_deep_seq/run_ws6.py --synth null     --out results/ws6_null/ --epochs 30
python workstreams/ws6_deep_seq/run_ws6.py --synth positive --out results/ws6_pos/  --epochs 30
```

**WS6.2 — free Colab T4 (recommended for the recurrent fits).** Open
`workstreams/ws6_deep_seq/colab_ws6.ipynb` (Runtime → T4 GPU); it installs torch, brings in the repo,
trains the O-view GRU on the T4 with the same code (`~1` hour), and prints the headline to paste back.
The AMD desktop GPU is **not** recommended (ROCm/DirectML caveat, §4).

**Outputs** (under `--out`, gitignored): per-`(view, target)` checkpoints
(`model_<world>_<target>_<view>.pt/.json` + a `.done` marker, fully resumable), standard-schema
predictions `pred_<target>_<world>_<view>_{val,test}.parquet`, `ws6_report_<world>.json`, and
`ws6_<world>.runmeta.json` (the SPEC §7 Pareto material). The exact formulas live in `THEORY.md`; the
exact code in `workstreams/ws6_deep_seq/model.py` and `run_ws6.py`; the plain-English tour in
`SEAN-README.md`.

In [ ]:
# Version stamp + run configuration (reproducibility).
import scipy, sklearn
print("python     ", platform.python_version())
print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
print("scipy      ", scipy.__version__)
print("scikit     ", sklearn.__version__)
print("matplotlib ", matplotlib.__version__)
print("torch      ", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device(auto):", resolve_device("auto"))
print("-" * 60)
print("DATA_MODE  ", DATA_MODE, " ->  run_ws6 synth =", _SYNTH[DATA_MODE])
print("views      ", VIEWS, " targets", TARGETS_)
print("arch       ", f"embed={EMBED} hidden={HIDDEN} max_len={MAX_LEN} batch={BATCH}")
print("budgets    ", f"n_games={NB_N_GAMES} epochs={NB_EPOCHS} n_perm={NB_N_PERM} "
      f"n_boot={NB_N_BOOT} ci_boot={NB_CI_BOOT}")
print("seed       ", SEED, " world_seed", WORLD_SEED)